In [ ]:
!pip install "tabpfn @ git+https://github.com/PriorLabs/TabPFN.git"

  Cloning https://github.com/PriorLabs/TabPFN.git to /tmp/pip-install-7gs9w6vc/tabpfn_cc0146f5054f49a791ea058470f99dba
  Running command git clone --filter=blob:none --quiet https://github.com/PriorLabs/TabPFN.git /tmp/pip-install-7gs9w6vc/tabpfn_cc0146f5054f49a791ea058470f99dba
  Resolved https://github.com/PriorLabs/TabPFN.git to commit 3a57521f4f3a6d2f6bdb7019c96c3d1ed2b88428
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.7/144.7 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 8.7 MB/s eta 0:00:00
  Created wheel for tabpfn: filename=tabpfn-6.4.1-py3-none-any.whl size=622383 sha256=7a60355084f750dd9a53d1297c186d8ef8ebbdc2a4d9bb478f61475e3ae91852
  Stored in directory: /tmp/pip-ephem-wheel-cache-q9_ch2x2/wheels/df/f4/05/8e4d074803d44fdeb5914b7e76831d3938b70128cb9bc93303
Successfully built tabpfn
  Attempting uninstal

In [ ]:
!hf auth login

# DEFAULT

### BANK

In [ ]:
!unzip archive_bank.zip

In [ ]:
import pandas as pd

bank_df = pd.read_csv("train.csv", sep=";")
bank_df

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,technician,married,tertiary,no,825,no,no,cellular,17,nov,977,3,-1,0,unknown,yes
45207,71,retired,divorced,primary,no,1729,no,no,cellular,17,nov,456,2,-1,0,unknown,yes
45208,72,retired,married,secondary,no,5715,no,no,cellular,17,nov,1127,5,184,3,success,yes
45209,57,blue-collar,married,secondary,no,668,no,no,telephone,17,nov,508,4,-1,0,unknown,no


In [ ]:
text_columns = bank_df.select_dtypes(include=['object']).columns
bank_df_encoded = pd.get_dummies(bank_df, columns=text_columns, drop_first=True)
bank_df_encoded

,age,balance,day,duration,campaign,pdays,previous,job_blue-collar,job_entrepreneur,job_housemaid,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,poutcome_unknown,y_yes
0,58,2143,5,261,1,-1,0,False,False,False,...,False,False,True,False,False,False,False,False,True,False
1,44,29,5,151,1,-1,0,False,False,False,...,False,False,True,False,False,False,False,False,True,False
2,33,2,5,76,1,-1,0,False,True,False,...,False,False,True,False,False,False,False,False,True,False
3,47,1506,5,92,1,-1,0,True,False,False,...,False,False,True,False,False,False,False,False,True,False
4,33,1,5,198,1,-1,0,False,False,False,...,False,False,True,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,825,17,977,3,-1,0,False,False,False,...,False,False,False,True,False,False,False,False,True,True
45207,71,1729,17,456,2,-1,0,False,False,False,...,False,False,False,True,False,False,False,False,True,True
45208,72,5715,17,1127,5,184,3,False,False,False,...,False,False,False,True,False,False,False,True,False,True
45209,57,668,17,508,4,-1,0,True,False,False,...,False,False,False,True,False,False,False,False,True,False


In [ ]:
%%time
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
import numpy as np

# Load data
X = bank_df_encoded.drop("y_yes", axis=1)
X = np.int32(X.to_numpy())
y = bank_df_encoded["y_yes"]
y = np.int32(y.to_numpy())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize a classifier
clf = TabPFNClassifier()  # Uses TabPFN 2.5 weights, finetuned on real data.
# To use TabPFN v2:
# clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)

CPU times: user 5.4 s, sys: 43 ms, total: 5.44 s
Wall time: 4.35 s


TabPFNClassifier()

In [ ]:
%%time
# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

ROC AUC: 0.9406863961034769
CPU times: user 3min 25s, sys: 3min 1s, total: 6min 26s
Wall time: 6min 33s


In [ ]:
%%time
# Predict labels
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))

Accuracy 0.910206789782152
CPU times: user 3min 27s, sys: 3min 2s, total: 6min 30s
Wall time: 6min 32s


In [ ]:
%%time
X = bank_df_encoded.drop("y_yes", axis=1)
X = np.int32(X.to_numpy())
y = bank_df_encoded["y_yes"]
y = np.int32(y.to_numpy())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = X_train[:64]
y_train = y_train[:64]

# Initialize a classifier
clf = TabPFNClassifier()  # Uses TabPFN 2.5 weights, finetuned on real data.
# To use TabPFN v2:
# clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)


# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

# Predict labels
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))

NameError: name 'bank_df_encoded' is not defined

In [ ]:
%%time
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
import numpy as np

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

# Load data
X = bank_df_encoded.drop("y_yes", axis=1)
X = np.int32(X.to_numpy())
y = bank_df_encoded["y_yes"]
y = np.int32(y.to_numpy())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Initialize a classifier
# clf = TabPFNClassifier()  # Uses TabPFN 2.5 weights, finetuned on real data.
# To use TabPFN v2:
clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.ignore_pretraining_limits = True
clf.fit(X_train, y_train)


# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

# Predict labels
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))

tabpfn-v2-classifier-finetuned-zk73skhh.(…):   0%|          | 0.00/29.0M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

ROC AUC: 0.9289721482456447
Accuracy 0.9067384252432911
CPU times: user 8min 34s, sys: 6min 8s, total: 14min 43s
Wall time: 15min


In [ ]:
%%time
X = bank_df_encoded.drop("y_yes", axis=1)
X = np.int32(X.to_numpy())
y = bank_df_encoded["y_yes"]
y = np.int32(y.to_numpy())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

X_train = X_train[:64]
y_train = y_train[:64]

# Initialize a classifier
# clf = TabPFNClassifier()  # Uses TabPFN 2.5 weights, finetuned on real data.
# To use TabPFN v2:
clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)


# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

# Predict labels
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))

ROC AUC: 0.8715019667482796
Accuracy 0.8873307971335044
CPU times: user 31.8 s, sys: 41.5 ms, total: 31.8 s
Wall time: 30.9 s


## BLOOD

In [ ]:
!wget https://archive.ics.uci.edu/static/public/176/blood+transfusion+service+center.zip
!unzip blood+transfusion+service+center.zip

--2026-03-02 12:46:15--  https://archive.ics.uci.edu/static/public/176/blood+transfusion+service+center.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘blood+transfusion+service+center.zip’

blood+transfusion+s     [ <=>                ]   3.85K  --.-KB/s    in 0s      

2026-03-02 12:46:16 (1.06 GB/s) - ‘blood+transfusion+service+center.zip’ saved [3938]

Archive:  blood+transfusion+service+center.zip
  inflating: transfusion.data        
  inflating: transfusion.names       


In [ ]:
from IPython.core.display import skip_doctest
import pandas as pd

blood_df = pd.read_csv('transfusion.data', sep=',', names=["recency", "frequency", "monetary", "time", "y"], skiprows=1)
print(len(blood_df))
blood_df

748


,recency,frequency,monetary,time,y
0,2,50,12500,98,1
1,0,13,3250,28,1
2,1,16,4000,35,1
3,2,20,5000,45,1
4,1,24,6000,77,0
...,...,...,...,...,...
743,23,2,500,38,0
744,21,2,500,52,0
745,23,3,750,62,0
746,39,1,250,39,0


In [ ]:
%%time
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

# Load data
X = blood_df.drop("y", axis=1)
X = np.int32(X.to_numpy())
y = blood_df["y"]
y = np.int32(y.to_numpy())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize a classifier
clf = TabPFNClassifier()  # Uses TabPFN 2.5 weights, finetuned on real data.
# To use TabPFN v2:
# clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)

tabpfn-v2.5-classifier-v2.5_default.ckpt:   0%|          | 0.00/42.9M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

CPU times: user 657 ms, sys: 163 ms, total: 819 ms
Wall time: 4.94 s


TabPFNClassifier()

In [ ]:
%%time
# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

# Predict labels
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))

ROC AUC: 0.7547237502989715
Accuracy 0.7733333333333333
CPU times: user 1.6 s, sys: 6.98 ms, total: 1.61 s
Wall time: 2.06 s


## CALHOUSING

In [ ]:
!pip install openml

In [ ]:
import openml as oml
dataset = oml.datasets.get_dataset(44090)

In [ ]:
df, _, _, _ = dataset.get_data()

In [ ]:
print(len(df))
df

In [ ]:
%%time
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

# Load data
X = df.drop("price", axis=1)
X = np.int32(X.to_numpy())
y = df["price"]
y = np.int32(y.to_numpy())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize a classifier
clf = TabPFNClassifier()  # Uses TabPFN 2.5 weights, finetuned on real data.
# To use TabPFN v2:
# clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)

# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

# Predict labels
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))

ROC AUC: 0.9425214187075501
Accuracy 0.8616428398352314
CPU times: user 42.6 s, sys: 3.1 s, total: 45.7 s
Wall time: 49.3 s


## CREDIT

In [ ]:
!wget https://archive.ics.uci.edu/static/public/144/statlog+german+credit+data.zip
!unzip statlog+german+credit+data.zip

--2026-03-02 12:49:00--  https://archive.ics.uci.edu/static/public/144/statlog+german+credit+data.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘statlog+german+credit+data.zip’

statlog+german+cred     [ <=>                ]  28.91K  --.-KB/s    in 0.008s  

2026-03-02 12:49:00 (3.49 MB/s) - ‘statlog+german+credit+data.zip’ saved [29607]

Archive:  statlog+german+credit+data.zip
  inflating: german.data             
  inflating: german.data-numeric     
  inflating: german.doc              
  inflating: Index                   


In [ ]:
import pandas as pd

df = pd.read_csv('german.data', sep=' ', names=['Attribute1', 'Attribute2', 'Attribute3', 'Attribute4', 'Attribute5', 'Attribute6', 'Attribute7', 'Attribute8', 'Attribute9', 'Attribute10', 'Attribute11', 'Attribute12', 'Attribute13', 'Attribute14', 'Attribute15', 'Attribute16', 'Attribute17', 'Attribute18', 'Attribute19', 'Attribute20', 'class'])
print(len(df))
df

1000


,Attribute1,Attribute2,Attribute3,Attribute4,Attribute5,Attribute6,Attribute7,Attribute8,Attribute9,Attribute10,...,Attribute12,Attribute13,Attribute14,Attribute15,Attribute16,Attribute17,Attribute18,Attribute19,Attribute20,class
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,A14,12,A32,A42,1736,A61,A74,3,A92,A101,...,A121,31,A143,A152,1,A172,1,A191,A201,1
996,A11,30,A32,A41,3857,A61,A73,4,A91,A101,...,A122,40,A143,A152,1,A174,1,A192,A201,1
997,A14,12,A32,A43,804,A61,A75,4,A93,A101,...,A123,38,A143,A152,1,A173,1,A191,A201,1
998,A11,45,A32,A43,1845,A61,A73,4,A93,A101,...,A124,23,A143,A153,1,A173,1,A192,A201,2


In [ ]:
text_columns = df.select_dtypes(include=['object']).columns
df_encoded = pd.get_dummies(df, columns=text_columns, drop_first=True)
df_encoded["class"] = df_encoded["class"] - 1
df_encoded

,Attribute2,Attribute5,Attribute8,Attribute11,Attribute13,Attribute16,Attribute18,class,Attribute1_A12,Attribute1_A13,...,Attribute12_A124,Attribute14_A142,Attribute14_A143,Attribute15_A152,Attribute15_A153,Attribute17_A172,Attribute17_A173,Attribute17_A174,Attribute19_A192,Attribute20_A202
0,6,1169,4,4,67,2,1,0,False,False,...,False,False,True,True,False,False,True,False,True,False
1,48,5951,2,2,22,1,1,1,True,False,...,False,False,True,True,False,False,True,False,False,False
2,12,2096,2,3,49,1,2,0,False,False,...,False,False,True,True,False,True,False,False,False,False
3,42,7882,2,4,45,1,2,0,False,False,...,False,False,True,False,True,False,True,False,False,False
4,24,4870,3,4,53,2,2,1,False,False,...,True,False,True,False,True,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,12,1736,3,4,31,1,1,0,False,False,...,False,False,True,True,False,True,False,False,False,False
996,30,3857,4,4,40,1,1,0,False,False,...,False,False,True,True,False,False,False,True,True,False
997,12,804,4,4,38,1,1,0,False,False,...,False,False,True,True,False,False,True,False,False,False
998,45,1845,4,4,23,1,1,1,False,False,...,True,False,True,False,True,False,True,False,True,False


In [ ]:
%%time
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

# Load data
X = df_encoded.drop("class", axis=1)
X = np.int32(X.to_numpy())
y = df_encoded["class"]
y = np.int32(y.to_numpy())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize a classifier
clf = TabPFNClassifier()  # Uses TabPFN 2.5 weights, finetuned on real data.
# To use TabPFN v2:
# clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)

# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

# Predict labels
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))

ROC AUC: 0.8329126096886645
Accuracy 0.795
CPU times: user 3.42 s, sys: 22.8 ms, total: 3.44 s
Wall time: 5.32 s


## DIABETES

In [ ]:
!wget https://github.com/npradaschnor/Pima-Indians-Diabetes-Dataset/raw/master/diabetes.csv
#!unzip statlog+german+credit+data.zip

--2026-03-02 12:50:46--  https://github.com/npradaschnor/Pima-Indians-Diabetes-Dataset/raw/master/diabetes.csv
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv [following]
--2026-03-02 12:50:47--  https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 23105 (23K) [text/plain]
Saving to: ‘diabetes.csv’

diabetes.csv        100%[===================>]  22.56K  --.-KB/s    in 0.002s  

2026-03-02 12:50:47 (13.3 MB/s) - ‘diabetes.csv’ saved [23105/231

In [ ]:
df = pd.read_csv('diabetes.csv')
print(len(df))
df

768


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [ ]:
%%time
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

# Load data
X = df.drop("Outcome", axis=1)
X = np.int32(X.to_numpy())
y = df["Outcome"]
y = np.int32(y.to_numpy())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize a classifier
clf = TabPFNClassifier()  # Uses TabPFN 2.5 weights, finetuned on real data.
# To use TabPFN v2:
# clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)

# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

# Predict labels
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))

ROC AUC: 0.8229568411386593
Accuracy 0.7532467532467533
CPU times: user 1.97 s, sys: 24.5 ms, total: 1.99 s
Wall time: 2.08 s


## HEART

In [ ]:
!unzip archive_heart.zip

Archive:  archive_heart.zip
  inflating: heart.csv               


In [ ]:
df = pd.read_csv('heart.csv')
print(len(df))
df

918


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0
...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat,1
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat,1
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat,1
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat,1


In [ ]:
text_columns = df.select_dtypes(include=['object']).columns
df_encoded = pd.get_dummies(df, columns=text_columns, drop_first=True)
df_encoded

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ExerciseAngina_Y,ST_Slope_Flat,ST_Slope_Up
0,40,140,289,0,172,0.0,0,True,True,False,False,True,False,False,False,True
1,49,160,180,0,156,1.0,1,False,False,True,False,True,False,False,True,False
2,37,130,283,0,98,0.0,0,True,True,False,False,False,True,False,False,True
3,48,138,214,0,108,1.5,1,False,False,False,False,True,False,True,True,False
4,54,150,195,0,122,0.0,0,True,False,True,False,True,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,110,264,0,132,1.2,1,True,False,False,True,True,False,False,True,False
914,68,144,193,1,141,3.4,1,True,False,False,False,True,False,False,True,False
915,57,130,131,0,115,1.2,1,True,False,False,False,True,False,True,True,False
916,57,130,236,0,174,0.0,1,False,True,False,False,False,False,False,True,False


In [ ]:
%%time
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

# Load data
X = df_encoded.drop("HeartDisease", axis=1)
X = np.int32(X.to_numpy())
y = df_encoded["HeartDisease"]
y = np.int32(y.to_numpy())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize a classifier
clf = TabPFNClassifier()  # Uses TabPFN 2.5 weights, finetuned on real data.
# To use TabPFN v2:
# clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)

# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

# Predict labels
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))

ROC AUC: 0.9418011894647408
Accuracy 0.8858695652173914
CPU times: user 1.55 s, sys: 21.4 ms, total: 1.57 s
Wall time: 3.28 s


## INCOME

In [ ]:
!wget https://archive.ics.uci.edu/static/public/20/census+income.zip


--2026-03-02 12:53:07--  https://archive.ics.uci.edu/static/public/20/census+income.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘census+income.zip’

census+income.zip       [ <=>                ] 650.11K  --.-KB/s    in 0.04s   

2026-03-02 12:53:07 (14.4 MB/s) - ‘census+income.zip’ saved [665715]



In [ ]:
!unzip census+income.zip

Archive:  census+income.zip
  inflating: adult.data              
  inflating: adult.names             
  inflating: adult.test              
replace Index? [y]es, [n]o, [A]ll, [N]one, [r]ename: у
error:  invalid response [у]
replace Index? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: Index                   
  inflating: old.adult.names         


In [ ]:
import pandas as pd

df = pd.read_csv('adult.data', names = ['age','workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'])
print(len(df))
df

32561


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32557,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32558,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K
32559,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,<=50K


In [ ]:
text_columns = df.select_dtypes(include=['object']).columns
df_encoded = pd.get_dummies(df, columns=text_columns, drop_first=True)
df_encoded

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week,workclass_ Federal-gov,workclass_ Local-gov,workclass_ Never-worked,workclass_ Private,...,native-country_ Puerto-Rico,native-country_ Scotland,native-country_ South,native-country_ Taiwan,native-country_ Thailand,native-country_ Trinadad&Tobago,native-country_ United-States,native-country_ Vietnam,native-country_ Yugoslavia,income_ >50K
0,39,77516,13,2174,0,40,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
1,50,83311,13,0,0,13,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
2,38,215646,9,0,0,40,False,False,False,True,...,False,False,False,False,False,False,True,False,False,False
3,53,234721,7,0,0,40,False,False,False,True,...,False,False,False,False,False,False,True,False,False,False
4,28,338409,13,0,0,40,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,257302,12,0,0,38,False,False,False,True,...,False,False,False,False,False,False,True,False,False,False
32557,40,154374,9,0,0,40,False,False,False,True,...,False,False,False,False,False,False,True,False,False,True
32558,58,151910,9,0,0,40,False,False,False,True,...,False,False,False,False,False,False,True,False,False,False
32559,22,201490,9,0,0,20,False,False,False,True,...,False,False,False,False,False,False,True,False,False,False


In [ ]:
%%time
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

# Load data
X = df_encoded.drop("income_ >50K", axis=1)
X = np.int32(X.to_numpy())
y = df_encoded["income_ >50K"]
y = np.int32(y.to_numpy())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize a classifier
clf = TabPFNClassifier()  # Uses TabPFN 2.5 weights, finetuned on real data.
# To use TabPFN v2:
# clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)

# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

# Predict labels
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))

ROC AUC: 0.926573136995127
Accuracy 0.8734838016275142
CPU times: user 8min 16s, sys: 7min, total: 15min 16s
Wall time: 15min 21s


# Missing values

### BANK


In [ ]:
import numpy as np

bank_df_missing = bank_df_encoded.copy()

for col in bank_df_missing.select_dtypes(include=['bool']).columns:
    bank_df_missing[col] = bank_df_missing[col].astype(int)

total_cells = bank_df_missing.size
missing_percentage = 0.20
num_missing_values = int(total_cells * missing_percentage)

missing_row_indices = np.random.randint(0, bank_df_missing.shape[0], num_missing_values)
missing_col_indices = np.random.randint(0, bank_df_missing.shape[1], num_missing_values)

for i in range(num_missing_values):
    bank_df_missing.iloc[missing_row_indices[i], missing_col_indices[i]] = np.nan

bank_df_imputed = bank_df_missing.fillna(bank_df_missing.mean(numeric_only=True))

print(f"Original non-NaN count: {bank_df_encoded.notna().sum().sum()}")
print(f"Missing values introduced: {bank_df_missing.isna().sum().sum()}")
print(f"Remaining missing values after imputation: {bank_df_imputed.isna().sum().sum()}")
bank_df_imputed.head()

Original non-NaN count: 1944073
Missing values introduced: 352482
Remaining missing values after imputation: 0


,age,balance,day,duration,campaign,pdays,previous,job_blue-collar,job_entrepreneur,job_housemaid,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,poutcome_unknown,y_yes
0,58.000000,2143.000000,15.795954,261.000000,1.000000,-1.000000,0.000000,0.0,0.0,0.0,...,0.0,0.00000,1.0,0.00000,0.0,0.013168,0.0,0.033499,1.000000,0.117592
1,40.936782,1367.908668,5.000000,258.092366,2.753685,-1.000000,0.000000,0.0,0.0,0.0,...,0.0,0.00000,1.0,0.00000,0.0,0.000000,0.0,0.000000,0.817481,0.000000
2,33.000000,2.000000,5.000000,76.000000,1.000000,40.306888,0.581394,0.0,1.0,0.0,...,0.0,0.00000,1.0,0.00000,0.0,0.000000,0.0,0.000000,1.000000,0.000000
3,47.000000,1506.000000,5.000000,92.000000,2.753685,-1.000000,0.000000,1.0,0.0,0.0,...,0.0,0.01039,1.0,0.08791,0.0,0.013168,0.0,0.033499,0.817481,0.000000
4,33.000000,1.000000,5.000000,198.000000,1.000000,-1.000000,0.000000,0.0,0.0,0.0,...,0.0,0.00000,1.0,0.08791,0.0,0.000000,0.0,0.033499,0.817481,0.117592


In [ ]:
%%time
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
import numpy as np

# Load data from the imputed DataFrame
X = bank_df_imputed.drop("y_yes", axis=1)
X = np.int32(X.to_numpy())
y = bank_df_imputed["y_yes"]
y = np.int32(y.to_numpy())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize a classifier
clf = TabPFNClassifier()  # Uses TabPFN 2.5 weights, finetuned on real data.
# To use TabPFN v2:
# clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)

# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

# Predict labels
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))

tabpfn-v2.5-classifier-v2.5_default.ckpt:   0%|          | 0.00/42.9M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

ROC AUC: 0.8912210673792369


KeyboardInterrupt: 

In [ ]:
import numpy as np

bank_df_missing = bank_df_encoded.copy()

for col in bank_df_missing.select_dtypes(include=['bool']).columns:
    bank_df_missing[col] = bank_df_missing[col].astype(int)

total_cells = bank_df_missing.size
missing_percentage = 0.50
num_missing_values = int(total_cells * missing_percentage)

missing_row_indices = np.random.randint(0, bank_df_missing.shape[0], num_missing_values)
missing_col_indices = np.random.randint(0, bank_df_missing.shape[1], num_missing_values)

for i in range(num_missing_values):
    bank_df_missing.iloc[missing_row_indices[i], missing_col_indices[i]] = np.nan

bank_df_imputed = bank_df_missing.fillna(bank_df_missing.mean(numeric_only=True))

print(f"Original non-NaN count: {bank_df_encoded.notna().sum().sum()}")
print(f"Missing values introduced: {bank_df_missing.isna().sum().sum()}")
print(f"Remaining missing values after imputation: {bank_df_imputed.isna().sum().sum()}")
bank_df_imputed.head()

Original non-NaN count: 1944073
Missing values introduced: 765410
Remaining missing values after imputation: 0


,age,balance,day,duration,campaign,pdays,previous,job_blue-collar,job_entrepreneur,job_housemaid,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,poutcome_unknown,y_yes
0,40.908929,2143.000000,5.000000,261.000000,1.00000,39.8909,0.000000,0.0,0.000000,0.027573,...,0.118869,0.000000,1.000000,0.088961,0.000000,0.000000,0.000000,0.000000,0.815469,0.11576
1,40.908929,1360.318783,5.000000,151.000000,1.00000,39.8909,0.000000,0.0,0.000000,0.000000,...,0.000000,0.010426,0.305577,0.088961,0.015549,0.012681,0.000000,0.033631,1.000000,0.00000
2,33.000000,2.000000,15.845624,76.000000,2.74324,-1.0000,0.574664,0.0,0.033852,0.000000,...,0.118869,0.010426,0.305577,0.000000,0.015549,0.000000,0.039363,0.033631,1.000000,0.11576
3,40.908929,1506.000000,15.845624,92.000000,2.74324,39.8909,0.000000,1.0,0.000000,0.000000,...,0.000000,0.010426,1.000000,0.088961,0.000000,0.000000,0.000000,0.000000,1.000000,0.11576
4,40.908929,1360.318783,15.845624,257.214385,1.00000,-1.0000,0.000000,0.0,0.033852,0.000000,...,0.118869,0.010426,1.000000,0.000000,0.015549,0.000000,0.039363,0.033631,1.000000,0.11576


In [ ]:
%%time
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
import numpy as np

# Load data from the imputed DataFrame
X = bank_df_imputed.drop("y_yes", axis=1)
X = np.int32(X.to_numpy())
y = bank_df_imputed["y_yes"]
y = np.int32(y.to_numpy())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize a classifier
clf = TabPFNClassifier()  # Uses TabPFN 2.5 weights, finetuned on real data.
# To use TabPFN v2:
# clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)

# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

# Predict labels
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))

ROC AUC: 0.8306123161464993
Accuracy 0.9281211987172399
CPU times: user 7min 2s, sys: 6min 19s, total: 13min 22s
Wall time: 13min 53s


In [ ]:
import numpy as np

bank_df_missing = bank_df_encoded.copy()

for col in bank_df_missing.select_dtypes(include=['bool']).columns:
    bank_df_missing[col] = bank_df_missing[col].astype(int)

total_cells = bank_df_missing.size
missing_percentage = 0.90
num_missing_values = int(total_cells * missing_percentage)

missing_row_indices = np.random.randint(0, bank_df_missing.shape[0], num_missing_values)
missing_col_indices = np.random.randint(0, bank_df_missing.shape[1], num_missing_values)

for i in range(num_missing_values):
    bank_df_missing.iloc[missing_row_indices[i], missing_col_indices[i]] = np.nan

bank_df_imputed = bank_df_missing.fillna(bank_df_missing.mean(numeric_only=True))

print(f"Original non-NaN count: {bank_df_encoded.notna().sum().sum()}")
print(f"Missing values introduced: {bank_df_missing.isna().sum().sum()}")
print(f"Remaining missing values after imputation: {bank_df_imputed.isna().sum().sum()}")
bank_df_imputed.head()

Original non-NaN count: 1944073
Missing values introduced: 1154125
Remaining missing values after imputation: 0


,age,balance,day,duration,campaign,pdays,previous,job_blue-collar,job_entrepreneur,job_housemaid,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,poutcome_unknown,y_yes
0,40.909908,2143.000000,15.730515,261.000000,2.773195,-1.000000,0.000000,0.215823,0.000000,0.028529,...,0.119015,0.010613,1.00000,0.000000,0.016653,0.000000,0.039527,0.033782,0.818531,0.000000
1,44.000000,29.000000,15.730515,151.000000,2.773195,39.418837,0.000000,0.000000,0.034236,0.028529,...,0.119015,0.010613,0.30342,0.084385,0.000000,0.013395,0.039527,0.000000,1.000000,0.000000
2,40.909908,1372.246516,15.730515,258.408348,2.773195,-1.000000,0.000000,0.215823,0.034236,0.028529,...,0.119015,0.000000,1.00000,0.084385,0.000000,0.013395,0.000000,0.033782,0.818531,0.118927
3,47.000000,1506.000000,5.000000,92.000000,2.773195,-1.000000,0.000000,0.215823,0.034236,0.000000,...,0.000000,0.000000,0.30342,0.000000,0.016653,0.000000,0.000000,0.033782,1.000000,0.000000
4,40.909908,1.000000,5.000000,198.000000,1.000000,39.418837,0.587198,0.215823,0.000000,0.000000,...,0.000000,0.000000,0.30342,0.000000,0.000000,0.013395,0.000000,0.033782,1.000000,0.118927


In [ ]:
%%time
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
import numpy as np

# Load data from the imputed DataFrame
X = bank_df_imputed.drop("y_yes", axis=1)
X = np.int32(X.to_numpy())
y = bank_df_imputed["y_yes"]
y = np.int32(y.to_numpy())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize a classifier
clf = TabPFNClassifier()  # Uses TabPFN 2.5 weights, finetuned on real data.
# To use TabPFN v2:
# clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)

# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

# Predict labels
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))

ROC AUC: 0.7680172047493872


KeyboardInterrupt: 

## BLOOD

In [ ]:
from IPython.core.display import skip_doctest
import pandas as pd

df = pd.read_csv('transfusion.data', sep=',', names=["recency", "frequency", "monetary", "time", "y"], skiprows=1)
print(len(df))

748


In [ ]:
# Create a copy of the df (CALHOUSING) DataFrame
df_missing = df.copy()

# Simulate 30% missing values
total_cells = df_missing.size
missing_percentage = 0.20
num_missing_values = int(total_cells * missing_percentage)

# Get all possible row and column indices
rows = np.random.randint(0, df_missing.shape[0], num_missing_values)
cols = np.random.randint(0, df_missing.shape[1], num_missing_values)

# Introduce NaN values
for r, c in zip(rows, cols):
    df_missing.iloc[r, c] = np.nan

# Print statistics about introduced missing values
original_non_nan_count = df.notna().sum().sum()
introduced_missing_count = df_missing.isna().sum().sum() - df.isna().sum().sum()

print(f"Original non-NaN count in CALHOUSING: {original_non_nan_count}")
print(f"Missing values introduced (approx 30%): {introduced_missing_count}")

# Impute these missing values using the mean of each column
df_imputed = df_missing.fillna(df_missing.mean(numeric_only=True))

# Print statistics about remaining missing values
remaining_missing_count = df_imputed.isna().sum().sum()
print(f"Remaining missing values after mean imputation: {remaining_missing_count}")

# Prepare and Evaluate TabPFN on Imputed CALHOUSING Data
# Load features (X) and target (y) from the imputed CALHOUSING DataFrame
X_imputed = df_imputed.drop("y", axis=1)
y_imputed = df_imputed["y"]

# Convert them to np.int32 arrays
X_imputed = np.int32(X_imputed.to_numpy())
y_imputed = np.int32(y_imputed.to_numpy())

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y_imputed, test_size=0.2, random_state=42)

# Train a TabPFNClassifier
clf = TabPFNClassifier()
clf.fit(X_train, y_train)

# Predict probabilities and labels on the test set
prediction_probabilities = clf.predict_proba(X_test)
# predictions = clf.predict(X_test)

# Print the ROC AUC and Accuracy scores
print("\n--- TabPFN Performance on Imputed CALHOUSING Data ---")
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))
# print("Accuracy:", accuracy_score(y_test, predictions))

Original non-NaN count in CALHOUSING: 3740
Missing values introduced (approx 30%): 668
Remaining missing values after mean imputation: 0

--- TabPFN Performance on Imputed CALHOUSING Data ---
ROC AUC: 0.6668618266978923


In [ ]:
# Create a copy of the df (CALHOUSING) DataFrame
df_missing = df.copy()

# Simulate 30% missing values
total_cells = df_missing.size
missing_percentage = 0.50
num_missing_values = int(total_cells * missing_percentage)

# Get all possible row and column indices
rows = np.random.randint(0, df_missing.shape[0], num_missing_values)
cols = np.random.randint(0, df_missing.shape[1], num_missing_values)

# Introduce NaN values
for r, c in zip(rows, cols):
    df_missing.iloc[r, c] = np.nan

# Print statistics about introduced missing values
original_non_nan_count = df.notna().sum().sum()
introduced_missing_count = df_missing.isna().sum().sum() - df.isna().sum().sum()

print(f"Original non-NaN count in CALHOUSING: {original_non_nan_count}")
print(f"Missing values introduced (approx 30%): {introduced_missing_count}")

# Impute these missing values using the mean of each column
df_imputed = df_missing.fillna(df_missing.mean(numeric_only=True))

# Print statistics about remaining missing values
remaining_missing_count = df_imputed.isna().sum().sum()
print(f"Remaining missing values after mean imputation: {remaining_missing_count}")

# Prepare and Evaluate TabPFN on Imputed CALHOUSING Data
# Load features (X) and target (y) from the imputed CALHOUSING DataFrame
X_imputed = df_imputed.drop("y", axis=1)
y_imputed = df_imputed["y"]

# Convert them to np.int32 arrays
X_imputed = np.int32(X_imputed.to_numpy())
y_imputed = np.int32(y_imputed.to_numpy())

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y_imputed, test_size=0.2, random_state=42)

# Train a TabPFNClassifier
clf = TabPFNClassifier()
clf.fit(X_train, y_train)

# Predict probabilities and labels on the test set
prediction_probabilities = clf.predict_proba(X_test)
# predictions = clf.predict(X_test)

# Print the ROC AUC and Accuracy scores
print("\n--- TabPFN Performance on Imputed CALHOUSING Data ---")
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))
# print("Accuracy:", accuracy_score(y_test, predictions))

Original non-NaN count in CALHOUSING: 3740
Missing values introduced (approx 30%): 1475
Remaining missing values after mean imputation: 0

--- TabPFN Performance on Imputed CALHOUSING Data ---
ROC AUC: 0.680140273163529


In [ ]:
# Create a copy of the df (CALHOUSING) DataFrame
df_missing = df.copy()

# Simulate 30% missing values
total_cells = df_missing.size
missing_percentage = 0.90
num_missing_values = int(total_cells * missing_percentage)

# Get all possible row and column indices
rows = np.random.randint(0, df_missing.shape[0], num_missing_values)
cols = np.random.randint(0, df_missing.shape[1], num_missing_values)

# Introduce NaN values
for r, c in zip(rows, cols):
    df_missing.iloc[r, c] = np.nan

# Print statistics about introduced missing values
original_non_nan_count = df.notna().sum().sum()
introduced_missing_count = df_missing.isna().sum().sum() - df.isna().sum().sum()

print(f"Original non-NaN count in CALHOUSING: {original_non_nan_count}")
print(f"Missing values introduced (approx 30%): {introduced_missing_count}")

# Impute these missing values using the mean of each column
df_imputed = df_missing.fillna(df_missing.mean(numeric_only=True))

# Print statistics about remaining missing values
remaining_missing_count = df_imputed.isna().sum().sum()
print(f"Remaining missing values after mean imputation: {remaining_missing_count}")

# Prepare and Evaluate TabPFN on Imputed CALHOUSING Data
# Load features (X) and target (y) from the imputed CALHOUSING DataFrame
X_imputed = df_imputed.drop("y", axis=1)
y_imputed = df_imputed["y"]

# Convert them to np.int32 arrays
X_imputed = np.int32(X_imputed.to_numpy())
y_imputed = np.int32(y_imputed.to_numpy())

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y_imputed, test_size=0.2, random_state=42)

# Train a TabPFNClassifier
clf = TabPFNClassifier()
clf.fit(X_train, y_train)

# Predict probabilities and labels on the test set
prediction_probabilities = clf.predict_proba(X_test)
# predictions = clf.predict(X_test)

# Print the ROC AUC and Accuracy scores
print("\n--- TabPFN Performance on Imputed CALHOUSING Data ---")
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))
# print("Accuracy:", accuracy_score(y_test, predictions))

Original non-NaN count in CALHOUSING: 3740
Missing values introduced (approx 30%): 2230
Remaining missing values after mean imputation: 0

--- TabPFN Performance on Imputed CALHOUSING Data ---
ROC AUC: 0.5236344537815126


## CALHOUSING

In [ ]:
!pip install openml

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.4/160.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 78.4 MB/s eta 0:00:00
  Created wheel for liac-arff: filename=liac_arff-2.5.0-py3-none-any.whl size=11717 sha256=ad5c17cb0db8726e8404bad26b5399494096217c613a2566eacd14dd7107eb97
  Stored in directory: /root/.cache/pip/wheels/a9/ac/cf/c2919807a5c623926d217c0a18eb5b457e5c19d242c3b5963a
Successfully built liac-arff


In [ ]:
import openml as oml
dataset = oml.datasets.get_dataset(44090)
df, _, _, _ = dataset.get_data()
df['price'] = np.int32(df['price'])

In [ ]:
# Create a copy of the df (CALHOUSING) DataFrame
df_missing = df.copy()

# Simulate 30% missing values
total_cells = df_missing.size
missing_percentage = 0.20
num_missing_values = int(total_cells * missing_percentage)

# Get all possible row and column indices
rows = np.random.randint(0, df_missing.shape[0], num_missing_values)
cols = np.random.randint(0, df_missing.shape[1], num_missing_values)

# Introduce NaN values
for r, c in zip(rows, cols):
    df_missing.iloc[r, c] = np.nan

# Print statistics about introduced missing values
original_non_nan_count = df.notna().sum().sum()
introduced_missing_count = df_missing.isna().sum().sum() - df.isna().sum().sum()

print(f"Original non-NaN count in CALHOUSING: {original_non_nan_count}")
print(f"Missing values introduced (approx 30%): {introduced_missing_count}")

# Impute these missing values using the mean of each column
df_imputed = df_missing.fillna(df_missing.mean(numeric_only=True))

# Print statistics about remaining missing values
remaining_missing_count = df_imputed.isna().sum().sum()
print(f"Remaining missing values after mean imputation: {remaining_missing_count}")

# Prepare and Evaluate TabPFN on Imputed CALHOUSING Data
# Load features (X) and target (y) from the imputed CALHOUSING DataFrame
X_imputed = df_imputed.drop("price", axis=1)
y_imputed = df_imputed["price"]

# Convert them to np.int32 arrays
X_imputed = np.int32(X_imputed.to_numpy())
y_imputed = np.int32(y_imputed.to_numpy())

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y_imputed, test_size=0.2, random_state=42)

# Train a TabPFNClassifier
clf = TabPFNClassifier()
clf.fit(X_train, y_train)

# Predict probabilities and labels on the test set
prediction_probabilities = clf.predict_proba(X_test)
# predictions = clf.predict(X_test)

# Print the ROC AUC and Accuracy scores
print("\n--- TabPFN Performance on Imputed CALHOUSING Data ---")
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))
# print("Accuracy:", accuracy_score(y_test, predictions))

Original non-NaN count in CALHOUSING: 185706
Missing values introduced (approx 30%): 33669
Remaining missing values after mean imputation: 0

--- TabPFN Performance on Imputed CALHOUSING Data ---
ROC AUC: 0.8376516160182598


In [ ]:
# Create a copy of the df (CALHOUSING) DataFrame
df_missing = df.copy()

# Simulate 30% missing values
total_cells = df_missing.size
missing_percentage = 0.50
num_missing_values = int(total_cells * missing_percentage)

# Get all possible row and column indices
rows = np.random.randint(0, df_missing.shape[0], num_missing_values)
cols = np.random.randint(0, df_missing.shape[1], num_missing_values)

# Introduce NaN values
for r, c in zip(rows, cols):
    df_missing.iloc[r, c] = np.nan

# Print statistics about introduced missing values
original_non_nan_count = df.notna().sum().sum()
introduced_missing_count = df_missing.isna().sum().sum() - df.isna().sum().sum()

print(f"Original non-NaN count in CALHOUSING: {original_non_nan_count}")
print(f"Missing values introduced (approx 30%): {introduced_missing_count}")

# Impute these missing values using the mean of each column
df_imputed = df_missing.fillna(df_missing.mean(numeric_only=True))

# Print statistics about remaining missing values
remaining_missing_count = df_imputed.isna().sum().sum()
print(f"Remaining missing values after mean imputation: {remaining_missing_count}")

# Prepare and Evaluate TabPFN on Imputed CALHOUSING Data
# Load features (X) and target (y) from the imputed CALHOUSING DataFrame
X_imputed = df_imputed.drop("price", axis=1)
y_imputed = df_imputed["price"]

# Convert them to np.int32 arrays
X_imputed = np.int32(X_imputed.to_numpy())
y_imputed = np.int32(y_imputed.to_numpy())

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y_imputed, test_size=0.2, random_state=42)

# Train a TabPFNClassifier
clf = TabPFNClassifier()
clf.fit(X_train, y_train)

# Predict probabilities and labels on the test set
prediction_probabilities = clf.predict_proba(X_test)
# predictions = clf.predict(X_test)

# Print the ROC AUC and Accuracy scores
print("\n--- TabPFN Performance on Imputed CALHOUSING Data ---")
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))
# print("Accuracy:", accuracy_score(y_test, predictions))

Original non-NaN count in CALHOUSING: 185706
Missing values introduced (approx 30%): 72961
Remaining missing values after mean imputation: 0

--- TabPFN Performance on Imputed CALHOUSING Data ---
ROC AUC: 0.7596354301246111


In [ ]:
# Create a copy of the df (CALHOUSING) DataFrame
df_missing = df.copy()

# Simulate 30% missing values
total_cells = df_missing.size
missing_percentage = 0.90
num_missing_values = int(total_cells * missing_percentage)

# Get all possible row and column indices
rows = np.random.randint(0, df_missing.shape[0], num_missing_values)
cols = np.random.randint(0, df_missing.shape[1], num_missing_values)

# Introduce NaN values
for r, c in zip(rows, cols):
    df_missing.iloc[r, c] = np.nan

# Print statistics about introduced missing values
original_non_nan_count = df.notna().sum().sum()
introduced_missing_count = df_missing.isna().sum().sum() - df.isna().sum().sum()

print(f"Original non-NaN count in CALHOUSING: {original_non_nan_count}")
print(f"Missing values introduced (approx 30%): {introduced_missing_count}")

# Impute these missing values using the mean of each column
df_imputed = df_missing.fillna(df_missing.mean(numeric_only=True))

# Print statistics about remaining missing values
remaining_missing_count = df_imputed.isna().sum().sum()
print(f"Remaining missing values after mean imputation: {remaining_missing_count}")

# Prepare and Evaluate TabPFN on Imputed CALHOUSING Data
# Load features (X) and target (y) from the imputed CALHOUSING DataFrame
X_imputed = df_imputed.drop("price", axis=1)
y_imputed = df_imputed["price"]

# Convert them to np.int32 arrays
X_imputed = np.int32(X_imputed.to_numpy())
y_imputed = np.int32(y_imputed.to_numpy())

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y_imputed, test_size=0.2, random_state=42)

# Train a TabPFNClassifier
clf = TabPFNClassifier()
clf.fit(X_train, y_train)

# Predict probabilities and labels on the test set
prediction_probabilities = clf.predict_proba(X_test)
# predictions = clf.predict(X_test)

# Print the ROC AUC and Accuracy scores
print("\n--- TabPFN Performance on Imputed CALHOUSING Data ---")
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))
# print("Accuracy:", accuracy_score(y_test, predictions))

Original non-NaN count in CALHOUSING: 185706
Missing values introduced (approx 30%): 110192
Remaining missing values after mean imputation: 0

--- TabPFN Performance on Imputed CALHOUSING Data ---
ROC AUC: 0.6832635031127166


## CREDIT G

In [ ]:
import pandas as pd

df = pd.read_csv('german.data', sep=' ', names=['Attribute1', 'Attribute2', 'Attribute3', 'Attribute4', 'Attribute5', 'Attribute6', 'Attribute7', 'Attribute8', 'Attribute9', 'Attribute10', 'Attribute11', 'Attribute12', 'Attribute13', 'Attribute14', 'Attribute15', 'Attribute16', 'Attribute17', 'Attribute18', 'Attribute19', 'Attribute20', 'class'])
print(len(df))
text_columns = df.select_dtypes(include=['object']).columns
df_encoded = pd.get_dummies(df, columns=text_columns, drop_first=True)
df_encoded["class"] = df_encoded["class"] - 1
for col in df_encoded.select_dtypes(include=['bool']).columns:
    df_encoded[col] = df_encoded[col].astype(int)
df_encoded

1000


,Attribute2,Attribute5,Attribute8,Attribute11,Attribute13,Attribute16,Attribute18,class,Attribute1_A12,Attribute1_A13,...,Attribute12_A124,Attribute14_A142,Attribute14_A143,Attribute15_A152,Attribute15_A153,Attribute17_A172,Attribute17_A173,Attribute17_A174,Attribute19_A192,Attribute20_A202
0,6,1169,4,4,67,2,1,0,0,0,...,0,0,1,1,0,0,1,0,1,0
1,48,5951,2,2,22,1,1,1,1,0,...,0,0,1,1,0,0,1,0,0,0
2,12,2096,2,3,49,1,2,0,0,0,...,0,0,1,1,0,1,0,0,0,0
3,42,7882,2,4,45,1,2,0,0,0,...,0,0,1,0,1,0,1,0,0,0
4,24,4870,3,4,53,2,2,1,0,0,...,1,0,1,0,1,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,12,1736,3,4,31,1,1,0,0,0,...,0,0,1,1,0,1,0,0,0,0
996,30,3857,4,4,40,1,1,0,0,0,...,0,0,1,1,0,0,0,1,1,0
997,12,804,4,4,38,1,1,0,0,0,...,0,0,1,1,0,0,1,0,0,0
998,45,1845,4,4,23,1,1,1,0,0,...,1,0,1,0,1,0,1,0,1,0


In [ ]:
# Create a copy of the df (CALHOUSING) DataFrame
df_missing = df_encoded.copy()

# Simulate 30% missing values
total_cells = df_missing.size
missing_percentage = 0.20
num_missing_values = int(total_cells * missing_percentage)

# Get all possible row and column indices
rows = np.random.randint(0, df_missing.shape[0], num_missing_values)
cols = np.random.randint(0, df_missing.shape[1], num_missing_values)

# Introduce NaN values
for r, c in zip(rows, cols):
    df_missing.iloc[r, c] = np.nan

# Print statistics about introduced missing values
original_non_nan_count = df.notna().sum().sum()
introduced_missing_count = df_missing.isna().sum().sum() - df.isna().sum().sum()

print(f"Original non-NaN count in CALHOUSING: {original_non_nan_count}")
print(f"Missing values introduced (approx 30%): {introduced_missing_count}")

# Impute these missing values using the mean of each column
df_imputed = df_missing.fillna(df_missing.mean(numeric_only=True))

# Print statistics about remaining missing values
remaining_missing_count = df_imputed.isna().sum().sum()
print(f"Remaining missing values after mean imputation: {remaining_missing_count}")

# Prepare and Evaluate TabPFN on Imputed CALHOUSING Data
# Load features (X) and target (y) from the imputed CALHOUSING DataFrame
X_imputed = df_imputed.drop("class", axis=1)
y_imputed = df_imputed["class"]

# Convert them to np.int32 arrays
X_imputed = np.int32(X_imputed.to_numpy())
y_imputed = np.int32(y_imputed.to_numpy())

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y_imputed, test_size=0.2, random_state=42)

# Train a TabPFNClassifier
clf = TabPFNClassifier()
clf.fit(X_train, y_train)

# Predict probabilities and labels on the test set
prediction_probabilities = clf.predict_proba(X_test)
predictions = clf.predict(X_test)

# Print the ROC AUC and Accuracy scores
print("\n--- TabPFN Performance on Imputed CALHOUSING Data ---")
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))
print("Accuracy:", accuracy_score(y_test, predictions))

Original non-NaN count in CALHOUSING: 21000
Missing values introduced (approx 30%): 8825
Remaining missing values after mean imputation: 0

--- TabPFN Performance on Imputed CALHOUSING Data ---
ROC AUC: 0.7945348352106799
Accuracy: 0.785


In [ ]:
# Create a copy of the df (CALHOUSING) DataFrame
df_missing = df_encoded.copy()

# Simulate 30% missing values
total_cells = df_missing.size
missing_percentage = 0.50
num_missing_values = int(total_cells * missing_percentage)

# Get all possible row and column indices
rows = np.random.randint(0, df_missing.shape[0], num_missing_values)
cols = np.random.randint(0, df_missing.shape[1], num_missing_values)

# Introduce NaN values
for r, c in zip(rows, cols):
    df_missing.iloc[r, c] = np.nan

# Print statistics about introduced missing values
original_non_nan_count = df.notna().sum().sum()
introduced_missing_count = df_missing.isna().sum().sum() - df.isna().sum().sum()

print(f"Original non-NaN count in CALHOUSING: {original_non_nan_count}")
print(f"Missing values introduced (approx 30%): {introduced_missing_count}")

# Impute these missing values using the mean of each column
df_imputed = df_missing.fillna(df_missing.mean(numeric_only=True))

# Print statistics about remaining missing values
remaining_missing_count = df_imputed.isna().sum().sum()
print(f"Remaining missing values after mean imputation: {remaining_missing_count}")

# Prepare and Evaluate TabPFN on Imputed CALHOUSING Data
# Load features (X) and target (y) from the imputed CALHOUSING DataFrame
X_imputed = df_imputed.drop("class", axis=1)
y_imputed = df_imputed["class"]

# Convert them to np.int32 arrays
X_imputed = np.int32(X_imputed.to_numpy())
y_imputed = np.int32(y_imputed.to_numpy())

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y_imputed, test_size=0.2, random_state=42)

# Train a TabPFNClassifier
clf = TabPFNClassifier()
clf.fit(X_train, y_train)

# Predict probabilities and labels on the test set
prediction_probabilities = clf.predict_proba(X_test)
predictions = clf.predict(X_test)

# Print the ROC AUC and Accuracy scores
print("\n--- TabPFN Performance on Imputed CALHOUSING Data ---")
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))
print("Accuracy:", accuracy_score(y_test, predictions))

Original non-NaN count in CALHOUSING: 21000
Missing values introduced (approx 30%): 19362
Remaining missing values after mean imputation: 0

--- TabPFN Performance on Imputed CALHOUSING Data ---
ROC AUC: 0.6708341286505058
Accuracy: 0.845


In [ ]:
# Create a copy of the df (CALHOUSING) DataFrame
df_missing = df_encoded.copy()

# Simulate 30% missing values
total_cells = df_missing.size
missing_percentage = 0.90
num_missing_values = int(total_cells * missing_percentage)

# Get all possible row and column indices
rows = np.random.randint(0, df_missing.shape[0], num_missing_values)
cols = np.random.randint(0, df_missing.shape[1], num_missing_values)

# Introduce NaN values
for r, c in zip(rows, cols):
    df_missing.iloc[r, c] = np.nan

# Print statistics about introduced missing values
original_non_nan_count = df.notna().sum().sum()
introduced_missing_count = df_missing.isna().sum().sum() - df.isna().sum().sum()

print(f"Original non-NaN count in CALHOUSING: {original_non_nan_count}")
print(f"Missing values introduced (approx 30%): {introduced_missing_count}")

# Impute these missing values using the mean of each column
df_imputed = df_missing.fillna(df_missing.mean(numeric_only=True))

# Print statistics about remaining missing values
remaining_missing_count = df_imputed.isna().sum().sum()
print(f"Remaining missing values after mean imputation: {remaining_missing_count}")

# Prepare and Evaluate TabPFN on Imputed CALHOUSING Data
# Load features (X) and target (y) from the imputed CALHOUSING DataFrame
X_imputed = df_imputed.drop("class", axis=1)
y_imputed = df_imputed["class"]

# Convert them to np.int32 arrays
X_imputed = np.int32(X_imputed.to_numpy())
y_imputed = np.int32(y_imputed.to_numpy())

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y_imputed, test_size=0.2, random_state=42)

# Train a TabPFNClassifier
clf = TabPFNClassifier()
clf.fit(X_train, y_train)

# Predict probabilities and labels on the test set
prediction_probabilities = clf.predict_proba(X_test)
predictions = clf.predict(X_test)

# Print the ROC AUC and Accuracy scores
print("\n--- TabPFN Performance on Imputed CALHOUSING Data ---")
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))
print("Accuracy:", accuracy_score(y_test, predictions))

Original non-NaN count in CALHOUSING: 21000
Missing values introduced (approx 30%): 29005
Remaining missing values after mean imputation: 0

--- TabPFN Performance on Imputed CALHOUSING Data ---
ROC AUC: 0.6436781609195402
Accuracy: 0.855


## DIABETES

In [ ]:
df = pd.read_csv('diabetes.csv')
print(len(df))

768


In [ ]:
# Create a copy of the df (CALHOUSING) DataFrame
df_missing = df.copy()

# Simulate 30% missing values
total_cells = df_missing.size
missing_percentage = 0.20
num_missing_values = int(total_cells * missing_percentage)

# Get all possible row and column indices
rows = np.random.randint(0, df_missing.shape[0], num_missing_values)
cols = np.random.randint(0, df_missing.shape[1], num_missing_values)

# Introduce NaN values
for r, c in zip(rows, cols):
    df_missing.iloc[r, c] = np.nan

# Print statistics about introduced missing values
original_non_nan_count = df.notna().sum().sum()
introduced_missing_count = df_missing.isna().sum().sum() - df.isna().sum().sum()

print(f"Original non-NaN count in CALHOUSING: {original_non_nan_count}")
print(f"Missing values introduced (approx 30%): {introduced_missing_count}")

# Impute these missing values using the mean of each column
df_imputed = df_missing.fillna(df_missing.mean(numeric_only=True))

# Print statistics about remaining missing values
remaining_missing_count = df_imputed.isna().sum().sum()
print(f"Remaining missing values after mean imputation: {remaining_missing_count}")

# Prepare and Evaluate TabPFN on Imputed CALHOUSING Data
# Load features (X) and target (y) from the imputed CALHOUSING DataFrame
X_imputed = df_imputed.drop("Outcome", axis=1)
y_imputed = df_imputed["Outcome"]

# Convert them to np.int32 arrays
X_imputed = np.int32(X_imputed.to_numpy())
y_imputed = np.int32(y_imputed.to_numpy())

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y_imputed, test_size=0.2, random_state=42)

# Train a TabPFNClassifier
clf = TabPFNClassifier()
clf.fit(X_train, y_train)

# Predict probabilities and labels on the test set
prediction_probabilities = clf.predict_proba(X_test)
predictions = clf.predict(X_test)

# Print the ROC AUC and Accuracy scores
print("\n--- TabPFN Performance on Imputed CALHOUSING Data ---")
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))
print("Accuracy:", accuracy_score(y_test, predictions))

Original non-NaN count in CALHOUSING: 6912
Missing values introduced (approx 30%): 1254
Remaining missing values after mean imputation: 0

--- TabPFN Performance on Imputed CALHOUSING Data ---
ROC AUC: 0.7599920461324319
Accuracy: 0.7532467532467533


In [ ]:
# Create a copy of the df (CALHOUSING) DataFrame
df_missing = df.copy()

# Simulate 30% missing values
total_cells = df_missing.size
missing_percentage = 0.50
num_missing_values = int(total_cells * missing_percentage)

# Get all possible row and column indices
rows = np.random.randint(0, df_missing.shape[0], num_missing_values)
cols = np.random.randint(0, df_missing.shape[1], num_missing_values)

# Introduce NaN values
for r, c in zip(rows, cols):
    df_missing.iloc[r, c] = np.nan

# Print statistics about introduced missing values
original_non_nan_count = df.notna().sum().sum()
introduced_missing_count = df_missing.isna().sum().sum() - df.isna().sum().sum()

print(f"Original non-NaN count in CALHOUSING: {original_non_nan_count}")
print(f"Missing values introduced (approx 30%): {introduced_missing_count}")

# Impute these missing values using the mean of each column
df_imputed = df_missing.fillna(df_missing.mean(numeric_only=True))

# Print statistics about remaining missing values
remaining_missing_count = df_imputed.isna().sum().sum()
print(f"Remaining missing values after mean imputation: {remaining_missing_count}")

# Prepare and Evaluate TabPFN on Imputed CALHOUSING Data
# Load features (X) and target (y) from the imputed CALHOUSING DataFrame
X_imputed = df_imputed.drop("Outcome", axis=1)
y_imputed = df_imputed["Outcome"]

# Convert them to np.int32 arrays
X_imputed = np.int32(X_imputed.to_numpy())
y_imputed = np.int32(y_imputed.to_numpy())

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y_imputed, test_size=0.2, random_state=42)

# Train a TabPFNClassifier
clf = TabPFNClassifier()
clf.fit(X_train, y_train)

# Predict probabilities and labels on the test set
prediction_probabilities = clf.predict_proba(X_test)
predictions = clf.predict(X_test)

# Print the ROC AUC and Accuracy scores
print("\n--- TabPFN Performance on Imputed CALHOUSING Data ---")
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))
print("Accuracy:", accuracy_score(y_test, predictions))

Original non-NaN count in CALHOUSING: 6912
Missing values introduced (approx 30%): 2701
Remaining missing values after mean imputation: 0

--- TabPFN Performance on Imputed CALHOUSING Data ---
ROC AUC: 0.763235294117647
Accuracy: 0.7792207792207793


In [ ]:
# Create a copy of the df (CALHOUSING) DataFrame
df_missing = df.copy()

# Simulate 30% missing values
total_cells = df_missing.size
missing_percentage = 0.90
num_missing_values = int(total_cells * missing_percentage)

# Get all possible row and column indices
rows = np.random.randint(0, df_missing.shape[0], num_missing_values)
cols = np.random.randint(0, df_missing.shape[1], num_missing_values)

# Introduce NaN values
for r, c in zip(rows, cols):
    df_missing.iloc[r, c] = np.nan

# Print statistics about introduced missing values
original_non_nan_count = df.notna().sum().sum()
introduced_missing_count = df_missing.isna().sum().sum() - df.isna().sum().sum()

print(f"Original non-NaN count in CALHOUSING: {original_non_nan_count}")
print(f"Missing values introduced (approx 30%): {introduced_missing_count}")

# Impute these missing values using the mean of each column
df_imputed = df_missing.fillna(df_missing.mean(numeric_only=True))

# Print statistics about remaining missing values
remaining_missing_count = df_imputed.isna().sum().sum()
print(f"Remaining missing values after mean imputation: {remaining_missing_count}")

# Prepare and Evaluate TabPFN on Imputed CALHOUSING Data
# Load features (X) and target (y) from the imputed CALHOUSING DataFrame
X_imputed = df_imputed.drop("Outcome", axis=1)
y_imputed = df_imputed["Outcome"]

# Convert them to np.int32 arrays
X_imputed = np.int32(X_imputed.to_numpy())
y_imputed = np.int32(y_imputed.to_numpy())

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y_imputed, test_size=0.2, random_state=42)

# Train a TabPFNClassifier
clf = TabPFNClassifier()
clf.fit(X_train, y_train)

# Predict probabilities and labels on the test set
prediction_probabilities = clf.predict_proba(X_test)
predictions = clf.predict(X_test)

# Print the ROC AUC and Accuracy scores
print("\n--- TabPFN Performance on Imputed CALHOUSING Data ---")
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))
print("Accuracy:", accuracy_score(y_test, predictions))

Original non-NaN count in CALHOUSING: 6912
Missing values introduced (approx 30%): 4071
Remaining missing values after mean imputation: 0

--- TabPFN Performance on Imputed CALHOUSING Data ---
ROC AUC: 0.5988805970149254
Accuracy: 0.8636363636363636


## HEART

In [ ]:
df = pd.read_csv('heart.csv')
print(len(df))

text_columns = df.select_dtypes(include=['object']).columns
df_encoded = pd.get_dummies(df, columns=text_columns, drop_first=True)

for col in df_encoded.select_dtypes(include=['bool']).columns:
    df_encoded[col] = df_encoded[col].astype(int)
df_encoded

918


,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ExerciseAngina_Y,ST_Slope_Flat,ST_Slope_Up
0,40,140,289,0,172,0.0,0,1,1,0,0,1,0,0,0,1
1,49,160,180,0,156,1.0,1,0,0,1,0,1,0,0,1,0
2,37,130,283,0,98,0.0,0,1,1,0,0,0,1,0,0,1
3,48,138,214,0,108,1.5,1,0,0,0,0,1,0,1,1,0
4,54,150,195,0,122,0.0,0,1,0,1,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,110,264,0,132,1.2,1,1,0,0,1,1,0,0,1,0
914,68,144,193,1,141,3.4,1,1,0,0,0,1,0,0,1,0
915,57,130,131,0,115,1.2,1,1,0,0,0,1,0,1,1,0
916,57,130,236,0,174,0.0,1,0,1,0,0,0,0,0,1,0


In [ ]:
# Create a copy of the df (CALHOUSING) DataFrame
df_missing = df_encoded.copy()

# Simulate 30% missing values
total_cells = df_missing.size
missing_percentage = 0.20
num_missing_values = int(total_cells * missing_percentage)

# Get all possible row and column indices
rows = np.random.randint(0, df_missing.shape[0], num_missing_values)
cols = np.random.randint(0, df_missing.shape[1], num_missing_values)

# Introduce NaN values
for r, c in zip(rows, cols):
    df_missing.iloc[r, c] = np.nan

# Print statistics about introduced missing values
original_non_nan_count = df.notna().sum().sum()
introduced_missing_count = df_missing.isna().sum().sum() - df.isna().sum().sum()

print(f"Original non-NaN count in CALHOUSING: {original_non_nan_count}")
print(f"Missing values introduced (approx 30%): {introduced_missing_count}")

# Impute these missing values using the mean of each column
df_imputed = df_missing.fillna(df_missing.mean(numeric_only=True))

# Print statistics about remaining missing values
remaining_missing_count = df_imputed.isna().sum().sum()
print(f"Remaining missing values after mean imputation: {remaining_missing_count}")

# Prepare and Evaluate TabPFN on Imputed CALHOUSING Data
# Load features (X) and target (y) from the imputed CALHOUSING DataFrame
X_imputed = df_imputed.drop("HeartDisease", axis=1)
y_imputed = df_imputed["HeartDisease"]

# Convert them to np.int32 arrays
X_imputed = np.int32(X_imputed.to_numpy())
y_imputed = np.int32(y_imputed.to_numpy())

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y_imputed, test_size=0.2, random_state=42)

# Train a TabPFNClassifier
clf = TabPFNClassifier()
clf.fit(X_train, y_train)

# Predict probabilities and labels on the test set
prediction_probabilities = clf.predict_proba(X_test)
predictions = clf.predict(X_test)

# Print the ROC AUC and Accuracy scores
print("\n--- TabPFN Performance on Imputed CALHOUSING Data ---")
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))
print("Accuracy:", accuracy_score(y_test, predictions))

Original non-NaN count in CALHOUSING: 11016
Missing values introduced (approx 30%): 2675
Remaining missing values after mean imputation: 0

--- TabPFN Performance on Imputed CALHOUSING Data ---
ROC AUC: 0.8351648351648351
Accuracy: 0.7282608695652174


In [ ]:
# Create a copy of the df (CALHOUSING) DataFrame
df_missing = df_encoded.copy()

# Simulate 30% missing values
total_cells = df_missing.size
missing_percentage = 0.50
num_missing_values = int(total_cells * missing_percentage)

# Get all possible row and column indices
rows = np.random.randint(0, df_missing.shape[0], num_missing_values)
cols = np.random.randint(0, df_missing.shape[1], num_missing_values)

# Introduce NaN values
for r, c in zip(rows, cols):
    df_missing.iloc[r, c] = np.nan

# Print statistics about introduced missing values
original_non_nan_count = df.notna().sum().sum()
introduced_missing_count = df_missing.isna().sum().sum() - df.isna().sum().sum()

print(f"Original non-NaN count in CALHOUSING: {original_non_nan_count}")
print(f"Missing values introduced (approx 30%): {introduced_missing_count}")

# Impute these missing values using the mean of each column
df_imputed = df_missing.fillna(df_missing.mean(numeric_only=True))

# Print statistics about remaining missing values
remaining_missing_count = df_imputed.isna().sum().sum()
print(f"Remaining missing values after mean imputation: {remaining_missing_count}")

# Prepare and Evaluate TabPFN on Imputed CALHOUSING Data
# Load features (X) and target (y) from the imputed CALHOUSING DataFrame
X_imputed = df_imputed.drop("HeartDisease", axis=1)
y_imputed = df_imputed["HeartDisease"]

# Convert them to np.int32 arrays
X_imputed = np.int32(X_imputed.to_numpy())
y_imputed = np.int32(y_imputed.to_numpy())

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y_imputed, test_size=0.2, random_state=42)

# Train a TabPFNClassifier
clf = TabPFNClassifier()
clf.fit(X_train, y_train)

# Predict probabilities and labels on the test set
prediction_probabilities = clf.predict_proba(X_test)
predictions = clf.predict(X_test)

# Print the ROC AUC and Accuracy scores
print("\n--- TabPFN Performance on Imputed CALHOUSING Data ---")
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))
print("Accuracy:", accuracy_score(y_test, predictions))

Original non-NaN count in CALHOUSING: 11016
Missing values introduced (approx 30%): 5815
Remaining missing values after mean imputation: 0

--- TabPFN Performance on Imputed CALHOUSING Data ---
ROC AUC: 0.7191606072203087
Accuracy: 0.6684782608695652


In [ ]:
# Create a copy of the df (CALHOUSING) DataFrame
df_missing = df_encoded.copy()

# Simulate 30% missing values
total_cells = df_missing.size
missing_percentage = 0.90
num_missing_values = int(total_cells * missing_percentage)

# Get all possible row and column indices
rows = np.random.randint(0, df_missing.shape[0], num_missing_values)
cols = np.random.randint(0, df_missing.shape[1], num_missing_values)

# Introduce NaN values
for r, c in zip(rows, cols):
    df_missing.iloc[r, c] = np.nan

# Print statistics about introduced missing values
original_non_nan_count = df.notna().sum().sum()
introduced_missing_count = df_missing.isna().sum().sum() - df.isna().sum().sum()

print(f"Original non-NaN count in CALHOUSING: {original_non_nan_count}")
print(f"Missing values introduced (approx 30%): {introduced_missing_count}")

# Impute these missing values using the mean of each column
df_imputed = df_missing.fillna(df_missing.mean(numeric_only=True))

# Print statistics about remaining missing values
remaining_missing_count = df_imputed.isna().sum().sum()
print(f"Remaining missing values after mean imputation: {remaining_missing_count}")

# Prepare and Evaluate TabPFN on Imputed CALHOUSING Data
# Load features (X) and target (y) from the imputed CALHOUSING DataFrame
X_imputed = df_imputed.drop("HeartDisease", axis=1)
y_imputed = df_imputed["HeartDisease"]

# Convert them to np.int32 arrays
X_imputed = np.int32(X_imputed.to_numpy())
y_imputed = np.int32(y_imputed.to_numpy())

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y_imputed, test_size=0.2, random_state=42)

# Train a TabPFNClassifier
clf = TabPFNClassifier()
clf.fit(X_train, y_train)

# Predict probabilities and labels on the test set
prediction_probabilities = clf.predict_proba(X_test)
predictions = clf.predict(X_test)

# Print the ROC AUC and Accuracy scores
print("\n--- TabPFN Performance on Imputed CALHOUSING Data ---")
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))
print("Accuracy:", accuracy_score(y_test, predictions))

Original non-NaN count in CALHOUSING: 11016
Missing values introduced (approx 30%): 8689
Remaining missing values after mean imputation: 0

--- TabPFN Performance on Imputed CALHOUSING Data ---
ROC AUC: 0.6923076923076924
Accuracy: 0.7119565217391305


## INCOME

In [ ]:
import pandas as pd

df = pd.read_csv('adult.data', names = ['age','workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'])
print(len(df))

text_columns = df.select_dtypes(include=['object']).columns
df_encoded = pd.get_dummies(df, columns=text_columns, drop_first=True)
for col in df_encoded.select_dtypes(include=['bool']).columns:
    df_encoded[col] = df_encoded[col].astype(int)
df_encoded

32561


,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week,workclass_ Federal-gov,workclass_ Local-gov,workclass_ Never-worked,workclass_ Private,...,native-country_ Puerto-Rico,native-country_ Scotland,native-country_ South,native-country_ Taiwan,native-country_ Thailand,native-country_ Trinadad&Tobago,native-country_ United-States,native-country_ Vietnam,native-country_ Yugoslavia,income_ >50K
0,39,77516,13,2174,0,40,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,50,83311,13,0,0,13,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
2,38,215646,9,0,0,40,0,0,0,1,...,0,0,0,0,0,0,1,0,0,0
3,53,234721,7,0,0,40,0,0,0,1,...,0,0,0,0,0,0,1,0,0,0
4,28,338409,13,0,0,40,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,257302,12,0,0,38,0,0,0,1,...,0,0,0,0,0,0,1,0,0,0
32557,40,154374,9,0,0,40,0,0,0,1,...,0,0,0,0,0,0,1,0,0,1
32558,58,151910,9,0,0,40,0,0,0,1,...,0,0,0,0,0,0,1,0,0,0
32559,22,201490,9,0,0,20,0,0,0,1,...,0,0,0,0,0,0,1,0,0,0


In [ ]:
# Create a copy of the df (CALHOUSING) DataFrame
df_missing = df_encoded.copy()

# Simulate 30% missing values
total_cells = df_missing.size
missing_percentage = 0.20
num_missing_values = int(total_cells * missing_percentage)

# Get all possible row and column indices
rows = np.random.randint(0, df_missing.shape[0], num_missing_values)
cols = np.random.randint(0, df_missing.shape[1], num_missing_values)

# Introduce NaN values
for r, c in zip(rows, cols):
    df_missing.iloc[r, c] = np.nan

# Print statistics about introduced missing values
original_non_nan_count = df.notna().sum().sum()
introduced_missing_count = df_missing.isna().sum().sum() - df.isna().sum().sum()

print(f"Original non-NaN count in CALHOUSING: {original_non_nan_count}")
print(f"Missing values introduced (approx 30%): {introduced_missing_count}")

# Impute these missing values using the mean of each column
df_imputed = df_missing.fillna(df_missing.mean(numeric_only=True))

# Print statistics about remaining missing values
remaining_missing_count = df_imputed.isna().sum().sum()
print(f"Remaining missing values after mean imputation: {remaining_missing_count}")

# Prepare and Evaluate TabPFN on Imputed CALHOUSING Data
# Load features (X) and target (y) from the imputed CALHOUSING DataFrame
X_imputed = df_imputed.drop("income_ >50K", axis=1)
y_imputed = df_imputed["income_ >50K"]

# Convert them to np.int32 arrays
X_imputed = np.int32(X_imputed.to_numpy())
y_imputed = np.int32(y_imputed.to_numpy())

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y_imputed, test_size=0.2, random_state=42)

# Train a TabPFNClassifier
clf = TabPFNClassifier()
clf.fit(X_train, y_train)

# Predict probabilities and labels on the test set
prediction_probabilities = clf.predict_proba(X_test)
# predictions = clf.predict(X_test)

# Print the ROC AUC and Accuracy scores
print("\n--- TabPFN Performance on Imputed CALHOUSING Data ---")
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))
# print("Accuracy:", accuracy_score(y_test, predictions))

Original non-NaN count in CALHOUSING: 488415
Missing values introduced (approx 30%): 595793
Remaining missing values after mean imputation: 0

--- TabPFN Performance on Imputed CALHOUSING Data ---
ROC AUC: 0.8866720022344928


In [ ]:
# Create a copy of the df (CALHOUSING) DataFrame
df_missing = df_encoded.copy()

# Simulate 30% missing values
total_cells = df_missing.size
missing_percentage = 0.50
num_missing_values = int(total_cells * missing_percentage)

# Get all possible row and column indices
rows = np.random.randint(0, df_missing.shape[0], num_missing_values)
cols = np.random.randint(0, df_missing.shape[1], num_missing_values)

# Introduce NaN values
for r, c in zip(rows, cols):
    df_missing.iloc[r, c] = np.nan

# Print statistics about introduced missing values
original_non_nan_count = df.notna().sum().sum()
introduced_missing_count = df_missing.isna().sum().sum() - df.isna().sum().sum()

print(f"Original non-NaN count in CALHOUSING: {original_non_nan_count}")
print(f"Missing values introduced (approx 30%): {introduced_missing_count}")

# Impute these missing values using the mean of each column
df_imputed = df_missing.fillna(df_missing.mean(numeric_only=True))

# Print statistics about remaining missing values
remaining_missing_count = df_imputed.isna().sum().sum()
print(f"Remaining missing values after mean imputation: {remaining_missing_count}")

# Prepare and Evaluate TabPFN on Imputed CALHOUSING Data
# Load features (X) and target (y) from the imputed CALHOUSING DataFrame
X_imputed = df_imputed.drop("income_ >50K", axis=1)
y_imputed = df_imputed["income_ >50K"]

# Convert them to np.int32 arrays
X_imputed = np.int32(X_imputed.to_numpy())
y_imputed = np.int32(y_imputed.to_numpy())

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y_imputed, test_size=0.2, random_state=42)

# Train a TabPFNClassifier
clf = TabPFNClassifier()
clf.fit(X_train, y_train)

# Predict probabilities and labels on the test set
prediction_probabilities = clf.predict_proba(X_test)
# predictions = clf.predict(X_test)

# Print the ROC AUC and Accuracy scores
print("\n--- TabPFN Performance on Imputed CALHOUSING Data ---")
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))
# print("Accuracy:", accuracy_score(y_test, predictions))

Original non-NaN count in CALHOUSING: 488415
Missing values introduced (approx 30%): 1293781
Remaining missing values after mean imputation: 0

--- TabPFN Performance on Imputed CALHOUSING Data ---
ROC AUC: 0.8327028885176916


In [ ]:
# Create a copy of the df (CALHOUSING) DataFrame
df_missing = df_encoded.copy()

# Simulate 30% missing values
total_cells = df_missing.size
missing_percentage = 0.90
num_missing_values = int(total_cells * missing_percentage)

# Get all possible row and column indices
rows = np.random.randint(0, df_missing.shape[0], num_missing_values)
cols = np.random.randint(0, df_missing.shape[1], num_missing_values)

# Introduce NaN values
for r, c in zip(rows, cols):
    df_missing.iloc[r, c] = np.nan

# Print statistics about introduced missing values
original_non_nan_count = df.notna().sum().sum()
introduced_missing_count = df_missing.isna().sum().sum() - df.isna().sum().sum()

print(f"Original non-NaN count in CALHOUSING: {original_non_nan_count}")
print(f"Missing values introduced (approx 30%): {introduced_missing_count}")

# Impute these missing values using the mean of each column
df_imputed = df_missing.fillna(df_missing.mean(numeric_only=True))

# Print statistics about remaining missing values
remaining_missing_count = df_imputed.isna().sum().sum()
print(f"Remaining missing values after mean imputation: {remaining_missing_count}")

# Prepare and Evaluate TabPFN on Imputed CALHOUSING Data
# Load features (X) and target (y) from the imputed CALHOUSING DataFrame
X_imputed = df_imputed.drop("income_ >50K", axis=1)
y_imputed = df_imputed["income_ >50K"]

# Convert them to np.int32 arrays
X_imputed = np.int32(X_imputed.to_numpy())
y_imputed = np.int32(y_imputed.to_numpy())

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y_imputed, test_size=0.2, random_state=42)

# Train a TabPFNClassifier
clf = TabPFNClassifier()
clf.fit(X_train, y_train)

# Predict probabilities and labels on the test set
prediction_probabilities = clf.predict_proba(X_test)
# predictions = clf.predict(X_test)

# Print the ROC AUC and Accuracy scores
print("\n--- TabPFN Performance on Imputed CALHOUSING Data ---")
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))
# print("Accuracy:", accuracy_score(y_test, predictions))

Original non-NaN count in CALHOUSING: 488415
Missing values introduced (approx 30%): 1951186
Remaining missing values after mean imputation: 0

--- TabPFN Performance on Imputed CALHOUSING Data ---
ROC AUC: 0.7846214376895146


# 64 SAMPLES

### BANK

In [ ]:
import pandas as pd

bank_df = pd.read_csv("train.csv", sep=";")

text_columns = bank_df.select_dtypes(include=['object']).columns
bank_df_encoded = pd.get_dummies(bank_df, columns=text_columns, drop_first=True)

X = bank_df_encoded.drop("y_yes", axis=1)
X = np.int32(X.to_numpy())
y = bank_df_encoded["y_yes"]
y = np.int32(y.to_numpy())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = X_train[:64]
y_train = y_train[:64]

# Initialize a classifier
clf = TabPFNClassifier()  # Uses TabPFN 2.5 weights, finetuned on real data.
# To use TabPFN v2:
# clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)

prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))

ROC AUC: 0.8377442127559122
Accuracy 0.886873825058056


### BLOOD

In [ ]:
from IPython.core.display import skip_doctest
import pandas as pd

blood_df = pd.read_csv('transfusion.data', sep=',', names=["recency", "frequency", "monetary", "time", "y"], skiprows=1)
print(len(blood_df))


from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
import numpy as np


# Load data
X = blood_df.drop("y", axis=1)
X = np.int32(X.to_numpy())
y = blood_df["y"]
y = np.int32(y.to_numpy())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = X_train[:64]
y_train = y_train[:64]

# Initialize a classifier
clf = TabPFNClassifier()  # Uses TabPFN 2.5 weights, finetuned on real data.
# To use TabPFN v2:
# clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)

# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

# Predict labels
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))


748
ROC AUC: 0.7169337479071992
Accuracy 0.7533333333333333


### CALHOUSING

In [ ]:
!pip install openml

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.4/160.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 58.1 MB/s eta 0:00:00
  Created wheel for liac-arff: filename=liac_arff-2.5.0-py3-none-any.whl size=11717 sha256=15f381b8fe986c68c9d5dd1e71ac2d9ff57d6af62c86285468870fdbfba40e56
  Stored in directory: /root/.cache/pip/wheels/a9/ac/cf/c2919807a5c623926d217c0a18eb5b457e5c19d242c3b5963a
Successfully built liac-arff


In [ ]:
import openml as oml
dataset = oml.datasets.get_dataset(44090)
df, _, _, _ = dataset.get_data()

from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

# Load data
X = df.drop("price", axis=1)
X = np.int32(X.to_numpy())
y = df["price"]
y = np.int32(y.to_numpy())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = X_train[:64]
y_train = y_train[:64]

# Initialize a classifier
clf = TabPFNClassifier()  # Uses TabPFN 2.5 weights, finetuned on real data.
# To use TabPFN v2:
# clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)

# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

# Predict labels
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))


ROC AUC: 0.8667851506355156
Accuracy 0.7887085049672886


### CREDIT

In [ ]:
import pandas as pd

df = pd.read_csv('german.data', sep=' ', names=['Attribute1', 'Attribute2', 'Attribute3', 'Attribute4', 'Attribute5', 'Attribute6', 'Attribute7', 'Attribute8', 'Attribute9', 'Attribute10', 'Attribute11', 'Attribute12', 'Attribute13', 'Attribute14', 'Attribute15', 'Attribute16', 'Attribute17', 'Attribute18', 'Attribute19', 'Attribute20', 'class'])
print(len(df))

text_columns = df.select_dtypes(include=['object']).columns
df_encoded = pd.get_dummies(df, columns=text_columns, drop_first=True)
df_encoded["class"] = df_encoded["class"] - 1

from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

# Load data
X = df_encoded.drop("class", axis=1)
X = np.int32(X.to_numpy())
y = df_encoded["class"]
y = np.int32(y.to_numpy())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = X_train[:64]
y_train = y_train[:64]

# Initialize a classifier
clf = TabPFNClassifier()  # Uses TabPFN 2.5 weights, finetuned on real data.
# To use TabPFN v2:
# clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)

# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

# Predict labels
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))

1000
ROC AUC: 0.555775934607525
Accuracy 0.7


### DIABETES

In [ ]:
df = pd.read_csv('diabetes.csv')
print(len(df))

from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

# Load data
X = df.drop("Outcome", axis=1)
X = np.int32(X.to_numpy())
y = df["Outcome"]
y = np.int32(y.to_numpy())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = X_train[:64]
y_train = y_train[:64]

# Initialize a classifier
clf = TabPFNClassifier()  # Uses TabPFN 2.5 weights, finetuned on real data.
# To use TabPFN v2:
# clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)

# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

# Predict labels
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))

768
ROC AUC: 0.7068870523415978
Accuracy 0.6623376623376623


### HEART

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('heart.csv')
print(len(df))

text_columns = df.select_dtypes(include=['object']).columns
df_encoded = pd.get_dummies(df, columns=text_columns, drop_first=True)

from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

# Load data
X = df_encoded.drop("HeartDisease", axis=1)
X = np.int32(X.to_numpy())
y = df_encoded["HeartDisease"]
y = np.int32(y.to_numpy())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = X_train[:64]
y_train = y_train[:64]

# Initialize a classifier
clf = TabPFNClassifier()  # Uses TabPFN 2.5 weights, finetuned on real data.
# To use TabPFN v2:
# clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)

# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

# Predict labels
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))

918


tabpfn-v2.5-classifier-v2.5_default.ckpt:   0%|          | 0.00/42.9M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

ROC AUC: 0.9163126593033136
Accuracy 0.8586956521739131


### INCOME

In [ ]:
df = pd.read_csv('adult.data', names = ['age','workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'])
print(len(df))

text_columns = df.select_dtypes(include=['object']).columns
df_encoded = pd.get_dummies(df, columns=text_columns, drop_first=True)

from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

# Load data
X = df_encoded.drop("income_ >50K", axis=1)
X = np.int32(X.to_numpy())
y = df_encoded["income_ >50K"]
y = np.int32(y.to_numpy())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = X_train[:64]
y_train = y_train[:64]

# Initialize a classifier
clf = TabPFNClassifier()  # Uses TabPFN 2.5 weights, finetuned on real data.
# To use TabPFN v2:
# clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)

# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

# Predict labels
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))


32561
ROC AUC: 0.8825344718016065
Accuracy 0.8306463995086749


# RESULTS

In [ ]:
import pandas as pd

res_list = [
    ["0", 0.93, 0.75, 0.94, 0.83, 0.82, 0.94, 0.93],
    ["20", 0.89, 0.68, 0.84, 0.79, 0.76, 0.84, 0.89],
    ["50", 0.83, 0.67, 0.76, 0.67, 0.76, 0.72, 0.83],
    ["90", 0.77, 0.52, 0.68, 0.64, 0.60, 0.69, 0.78],
    ["64 samples", 0.84, 0.72, 0.87, 0.56, 0.71, 0.91, 0.88]
]

pd.DataFrame(res_list, columns = ["Missing percent", "BANK", "BLOOD", "CALHOUSING", "CREDIT_G", "DIABETES", "HEART", "INCOME"])

,Missing percent,BANK,BLOOD,CALHOUSING,CREDIT_G,DIABETES,HEART,INCOME
0,0,0.93,0.75,0.94,0.83,0.82,0.94,0.93
1,20,0.89,0.68,0.84,0.79,0.76,0.84,0.89
2,50,0.83,0.67,0.76,0.67,0.76,0.72,0.83
3,90,0.77,0.52,0.68,0.64,0.60,0.69,0.78
4,64 samples,0.84,0.72,0.87,0.56,0.71,0.91,0.88
